### Instalação das dependências

Os pacotes abaixo são necessários para criar a API de predição com FastAPI. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install fastapi uvicorn pydantic pandas joblib scikit-learn -q

# Código 1.2 — Dockerfile e API Mínima de Predição

**Objetivo:** Mostrar como criar um serviço de inferência em Python usando **FastAPI** e como empacotar essa API dentro de um container **Docker**.

### Passo 1: Estrutura do Dockerfile para Servir a API

O bloco abaixo exibe as instruções do `Dockerfile` necessárias para construir a imagem containerizada.

In [ ]:
# Conteúdo do Dockerfile
dockerfile_content = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY api.py pipeline_producao_v1.joblib ./

EXPOSE 8000

CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
"""
print("Instruções do Dockerfile:")
print(dockerfile_content)

**Entendendo as instruções do Dockerfile:**
- `FROM python:3.11-slim`: Define a imagem base oficial do Python 3.11 em uma versão enxuta (*slim*).
- `WORKDIR /app`: Define o diretório de trabalho padrão dentro do container onde os arquivos serão salvos.
- `COPY requirements.txt .`: Copia o arquivo de dependências do computador local para dentro do container.
- `RUN pip install --no-cache-dir -r requirements.txt`: Instala os pacotes Python sem guardar cache para reduzir o tamanho da imagem.
- `EXPOSE 8000`: Informa que o container escutará na porta 8000.
- `CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]`: Comando executado ao iniciar o container para subir o servidor Uvicorn.

### Passo 2: Dependências do Projeto (`requirements.txt`)

Lista de pacotes Python fundamentais para rodar o serviço FastAPI.

In [ ]:
# Dependências necessárias
requirements = """
fastapi
joblib
pandas
scikit-learn
uvicorn[standard]
"""
print("Arquivo requirements.txt:")
print(requirements)

### Passo 3: Implementação da API com FastAPI (`api.py`)

Definimos o contrato de entrada dos dados com `Pydantic`, os endpoints `/health` (verificação de integridade) e `/predict` (inferência).

In [ ]:
# Código da aplicação FastAPI (api.py)
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

# Inicializar aplicação FastAPI
app = FastAPI(title="API de Predição de Falha", version="1.0.0")

# Carregar o pipeline pré-treinado no início da aplicação
pipeline = joblib.load("pipeline_producao_v1.joblib")


# Esquema de validação dos dados de entrada
class EquipamentoInput(BaseModel):
    temperatura: float
    vibracao: float
    pressao: float
    rpm: float


# Endpoint de verificação de saúde do serviço
@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "healthy"}


# Endpoint principal para inferência do modelo
@app.post("/predict")
def predict(dados: EquipamentoInput) -> dict[str, float | str]:
    # Converter entrada para DataFrame com os nomes corretos de colunas
    entrada = pd.DataFrame([dados.model_dump()])
    
    # Calcular probabilidade de falha (classe 1)
    probabilidade = float(pipeline.predict_proba(entrada)[0, 1])
    
    # Definir regra de negócio / status
    status = "CRITICO" if probabilidade >= 0.7 else "OK"
    
    return {
        "probabilidade_falha": round(probabilidade, 4),
        "status": status,
    }

print("Código da API carregado com sucesso!")

**Entendendo as variáveis e componentes da API:**
- `FastAPI(title=..., version=...)`: Cria a instância principal do framework configurando título e versão da documentação Swagger.
- `EquipamentoInput(BaseModel)`: Classe Pydantic que valida os tipos de dados recebidos no corpo da requisição HTTP (JSON).
  - `temperatura: float`, `vibracao: float`, `pressao: float`, `rpm: float`: Atributos esperados com validação automática de tipo.
- `@app.get("/health")`: Endpoint HTTP GET para monitoramento básico de disponibilidade.
- `@app.post("/predict")`: Endpoint HTTP POST que recebe os dados de um equipamento e retorna a predição.
- `dados.model_dump()`: Converte o objeto Pydantic validado em um dicionário Python para criação do DataFrame.

---

## Conclusão da Aula — Dockerfile e API Mínima

Empacotar o pipeline em uma **API com FastAPI** e containerizar com **Docker** garante a portabilidade da solução. Isso permite que qualquer infraestrutura (Cloud, Kubernetes ou servidor local) execute o serviço preditivo sem problemas de incompatibilidade de ambiente.